# GQSAT / GATQSAT — reproduce evaluation results on Colab

Goal: re-run the **evaluation of already-trained checkpoints** (no retraining) and check the
numbers still match the committed `runs/*/*.tsv`, i.e. that porting to a modern stack did **not**
change the semantics.

**Strategy (reproduce-first):**
* `torch` / `torch-geometric` **modern** (2.x). The only code change vs the original is that
  `torch-scatter`/`torch-sparse` were replaced by `torch_geometric.utils.scatter` — a 1:1
  functional swap (branch `port/modern`). This removes the #1 Colab install pain.
* `gym` is **pinned to the original version** so the `MiniSATEnv` and its registration stay
  byte-for-byte untouched → the MDP semantics are identical *by construction*. Modernising gym
  (gymnasium, 5-tuple step, dropping the deprecated `env_specs` registry) is a **separate**
  step we do only after this reproduction is green.
* Validate **GraphQSAT first** (no attention → no `GATConv`), then **GATQSAT** with an explicit
  `state_dict` key check, because `evaluate.py` loads with `strict=False` and a `GATConv`
  parameter-name drift between PyG versions would silently drop the attention weights.

This repo lives in Google Drive (Insync), so once Drive is mounted the working copy — including
the `port/modern` edits, the trained `runs/` and the `data/` — is already present. No clone/push.

## 1. Mount Drive and locate the repo

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

import os, glob
# the Insync-synced copy of this repo (adjust if your Drive layout differs)
candidates = glob.glob('/content/gdrive/MyDrive/**/neuroSAT', recursive=True) + \
             glob.glob('/content/gdrive/My Drive/**/neuroSAT', recursive=True)
assert candidates, 'neuroSAT folder not found under your Drive — set REPO manually.'
REPO = candidates[0]
print('repo:', REPO)
%cd $REPO
!git -C GQSAT rev-parse --abbrev-ref HEAD   # expect: port/modern

## 2. Install the modern stack (+ pinned gym for env fidelity)

`torch-geometric` is installed **without** `torch-scatter`/`torch-sparse`. `gym` is pinned; if the
pinned install fights with NumPy 2.x on Colab, the cell falls back to `numpy<2`.

In [ ]:
# modern PyG (uses torch already shipped by Colab); no torch-scatter / torch-sparse needed
!pip -q install torch-geometric
!pip -q install PyYAML tensorboardX

# gym pinned to the original API so MiniSATEnv + its registry code stay untouched.
# gym 0.21 broke the install tooling, so prep setuptools/wheel first, then pin.
!pip -q install 'setuptools<66' 'wheel<0.39'
!pip -q install 'gym==0.20.0' || pip -q install 'gym==0.21.0'
# gym 0.20/0.21 predate NumPy 2.0; pin NumPy down if imports complain.
!pip -q install 'numpy<2'

import gym, torch, torch_geometric
print('gym', gym.__version__, '| torch', torch.__version__, '| pyg', torch_geometric.__version__)

## 3. Build the MiniSat gym extension (`_GymSolver.so`)

The committed `.so` is tied to a specific Python version, so rebuild it for this Colab runtime.
`PYTHON` is auto-detected and the matching dev headers + swig are installed.

In [ ]:
import sys
PYV = f'python{sys.version_info.major}.{sys.version_info.minor}'
print('building for', PYV)
!sudo apt-get -qq install -y swig zlib1g-dev {PYV}-dev >/dev/null
%cd $REPO/GQSAT/minisat
!make clean >/dev/null 2>&1 ; true
# (re)generate the SWIG wrapper, then build the shared lib and the python module
!swig -fastdispatch -c++ -python -py3 minisat/gym/GymSolver.i 2>/dev/null ; true
!make -s 2>&1 | tail -n 5
!make python-wrap PYTHON={PYV} 2>&1 | tail -n 15
import os; print('built:', os.path.exists('minisat/gym/_GymSolver.so'))
%cd $REPO/GQSAT

In [ ]:
# sanity: importing `minisat` triggers env registration; importing models triggers the PyG swap
import importlib, sys
sys.path.insert(0, os.path.join(REPO, 'GQSAT'))
import minisat            # registers sat-v0 (works on pinned gym 0.20/0.21)
from minisat.minisat.gym.GymSolver import sat   # noqa: F401  (forces the .so to load)
import gqsat.models, gqsat.learners, gqsat.utils
print('imports OK — env + native solver + modern-PyG models all load')

## 4. Reproduce **GraphQSAT** (no attention) — validates the whole pipeline

Pick a trained GraphQSAT run, a graph-coloring set and a decision cap, then run `evaluate.py`.
The `status.yaml` is printed so we can choose the right checkpoint (the one that produced the
committed `.tsv`).

In [ ]:
import yaml, glob, os

RUN      = 'runs/Dec08_08-39-57_e63e47f25457'   # GraphQSAT (use_attention: false)
DATASET  = 'flat50-115'                          # a graph-coloring set
CAP      = 500                                   # test_time_max_decisions_allowed
DATA_DIR = f'../data/graph-coloring/{DATASET}'

with open(os.path.join(RUN, 'status.yaml')) as f:
    status = yaml.load(f, Loader=yaml.Loader)
# surface anything that looks like a best/last checkpoint pointer
print({k: v for k, v in status.items() if 'model' in str(k).lower() or 'step' in str(k).lower() or 'best' in str(k).lower()})

ckpts = sorted(glob.glob(os.path.join(RUN, 'model_*.chkp')),
               key=lambda p: int(p.split('_')[-1].split('.')[0]))
CKPT = os.path.basename(ckpts[-1])   # default: highest-step checkpoint; override if status says otherwise
print('using checkpoint:', CKPT, '| total checkpoints:', len(ckpts))

In [ ]:
import subprocess, re

def run_eval(run_dir, ckpt, data_dir, cap):
    cmd = ['python3', 'evaluate.py',
           '--logdir', 'log', '--env-name', 'sat-v0', '--core-steps', '-1',
           '--eps-final', '0.0', '--eval-time-limit', '100000000000000', '--no_restarts',
           '--test_time_max_decisions_allowed', str(cap),
           '--eval-problems-paths', data_dir,
           '--model-dir', run_dir, '--model-checkpoint', ckpt]
    out = subprocess.run(cmd, capture_output=True, text=True)
    print(out.stdout[-1500:])
    if out.returncode != 0:
        print('STDERR:\n', out.stderr[-2000:])
    m = re.search(r'median_relative_score:\s*([0-9.eE+-]+)', out.stdout)
    return float(m.group(1)) if m else None

fresh_median_graphqsat = run_eval(RUN, CKPT, DATA_DIR, CAP)
print('\nfresh GraphQSAT MRIR:', fresh_median_graphqsat)

## 5. Reproduce **GATQSAT** (attention) — with explicit `state_dict` key check

Before trusting the eval, confirm the attention (`GATConv`) weights actually load on the modern PyG.
`evaluate.py` uses `strict=False`, so missing/unexpected keys would otherwise be hidden.

In [ ]:
import torch, os, glob
from gqsat.models import SATModel

GAT_RUN = 'runs/Nov13_03-55-51_c42e8ad320d8'   # GATQSAT (use_attention: true, heads: 3)
gat_ckpts = sorted(glob.glob(os.path.join(GAT_RUN, 'model_*.chkp')),
                   key=lambda p: int(p.split('_')[-1].split('.')[0]))
GAT_CKPT = os.path.basename(gat_ckpts[-1])

net = SATModel.load_from_yaml(os.path.join(GAT_RUN, 'model.yaml'))
sd  = torch.load(os.path.join(GAT_RUN, GAT_CKPT), map_location='cpu')
res = net.load_state_dict(sd, strict=False)
print('checkpoint:', GAT_CKPT)
print('missing keys   (in model, absent from ckpt):', res.missing_keys)
print('unexpected keys(in ckpt, absent from model):', res.unexpected_keys)
assert not res.missing_keys and not res.unexpected_keys, \
    'GATConv param-name drift — attention weights would not load; pin a matching PyG or remap keys.'
print('\nall parameters loaded cleanly — attention weights are in place')

In [ ]:
fresh_median_gatqsat = run_eval(GAT_RUN, GAT_CKPT, DATA_DIR, CAP)
print('\nfresh GATQSAT MRIR:', fresh_median_gatqsat)

## 6. Compare with the committed `.tsv` (semantic non-regression)

The committed result for `<dataset>-<model>-max<cap>.tsv` is the original-stack output. If the
fresh MRIR matches (within tolerance) for the **same checkpoint**, the port preserves semantics.
A mismatch most likely means a *different* checkpoint produced the committed `.tsv` — adjust
`CKPT`/`GAT_CKPT` and re-run.

In [ ]:
import csv, statistics, os

def committed_median(run_dir, dataset, model, cap):
    path = os.path.join(run_dir, f'{dataset}-{model}-max{cap}.tsv')
    if not os.path.exists(path):
        return None
    vals = []
    with open(path, newline='') as f:
        for row in csv.DictReader(f, delimiter='\t'):
            try:
                v = float(row['score'])
                if v == v:
                    vals.append(v)
            except (ValueError, KeyError):
                pass
    return statistics.median(vals) if vals else None

for name, run_dir, model, fresh in [
    ('GraphQSAT', RUN,     'graphqsat', fresh_median_graphqsat),
    ('GATQSAT',   GAT_RUN, 'gatqsat',   fresh_median_gatqsat),
]:
    ref = committed_median(run_dir, DATASET, model, CAP)
    if fresh is None or ref is None:
        print(f'{name:9s}  fresh={fresh}  committed={ref}  -> cannot compare')
        continue
    ok = abs(fresh - ref) <= 0.05 * max(1.0, abs(ref))
    print(f'{name:9s}  fresh={fresh:.3f}  committed={ref:.3f}  '
          f'{"PASS" if ok else "DIFF (try another checkpoint)"}')